# Goal 3 · Knowledge graph ontology

**Goal:** repair the broken commercial-property entity link, and author a new **marine-cargo** branch with its policy class and governed loss-ratio metric, so both lines traverse from entity to the right metric.

**File:** `src/main.py` (`seed()` only)  |  **Evidence:** `ontology_result.json`

### The graph model

```
(:Chunk)-[:MENTIONS]->(:Entity)-[:INSTANCE_OF]->(:OntologyClass)-[:SUBCLASS_OF]->(:OntologyClass {FinancialAsset})
                                                        ^
                               (:GovernedMetric)-[:COMPUTED_OVER]
```

The assistant answers "which metric applies to X?" by walking **entity → class → metric**. The graded evidence is exactly that traversal, for two entities: `ent_nexus_tower` and `ent_atlantic_freight`.

The ingestion helpers in `src/ingestion.py` are already correct:
- `ingest_ontology_class(lab, uri, name, description, parent_uri=None)` creates or updates a class, plus `SUBCLASS_OF` to its parent.
- `ingest_governed_metric(lab, name, description, calculation_template, target_class_uris, properties_used)` creates `COMPUTED_OVER` edges, but only to classes that **already exist**.
- `dual_ingest_document_chunk(lab, os_store, chunk_id, text, source_uri, extracted_entities, pg_store=None)`:
  - embeds the text into OpenSearch (and pgvector);
  - adds the `Chunk` and `MENTIONS` → `Entity` to Neo4j;
  - adds `INSTANCE_OF` **only when the entity carries a `class_uri`**.

> **Where the chunks go:** `dual_ingest_document_chunk` writes each chunk to **two vector stores**:
> - OpenSearch Serverless, walked through in [`08_opensearch_walkthrough`](08_opensearch_walkthrough.ipynb);
> - Aurora pgvector, walked through in [`07_pgvector_walkthrough`](07_pgvector_walkthrough.ipynb).
>
> Both are optional and use scratch indexes or tables, so they don't disturb your evidence. They're most useful before goal 4.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('.'))
import labkit
from src.graph_client import GraphLabDriver
from src.emit_evidence import _ONTOLOGY_QUERY

lab = GraphLabDriver()

def cypher(q, **params):
    with lab.driver.session(default_access_mode='r') as s:
        return [r.data() for r in s.run(q, **params)]

print('The graded traversal:\n', _ONTOLOGY_QUERY)
for row in cypher(_ONTOLOGY_QUERY):
    print(row)

### Observe

- Nexus Tower exists, but its `class_name` is `None` and it has no metrics. Atlantic Freight doesn't exist at all.
- The next cell shows what *is* in the graph, and the line of ingestion Cypher that decides whether an entity gets linked.

In [ ]:
for row in cypher('MATCH (c:OntologyClass) OPTIONAL MATCH (c)-[:SUBCLASS_OF]->(p) OPTIONAL MATCH (m:GovernedMetric)-[:COMPUTED_OVER]->(c) '
                  'RETURN c.uri AS class, p.uri AS parent, collect(m.name) AS metrics'):
    print(row)
print(cypher('MATCH (k:Chunk)-[r:MENTIONS]->(e:Entity) OPTIONAL MATCH (e)-[:INSTANCE_OF]->(c) RETURN k.id AS chunk, e.id AS entity, e.type AS type, c.uri AS class'))
print()
labkit.show('src/ingestion.py', 'WHERE ent.class_uri IS NOT NULL', before=6, after=4)

## Task 3 · Edit `seed()` in `src/main.py`

1. **Repair the link.** Give the Nexus Tower entity (in the `doc_chunk_101` ingestion) a `class_uri` of `urn:coa:ontology:CommercialPropertyPolicy`.
2. **New class** `MarineCargoPolicy`:
   - URI `urn:coa:ontology:MarineCargoPolicy`, parent `urn:coa:ontology:FinancialAsset`;
   - description: *Insurance policies covering goods in transit by sea and related cargo liabilities.*
3. **New governed metric** `cargo_loss_ratio`, computed over `MarineCargoPolicy`:
   - description: *Loss ratio for marine cargo lines: incurred losses divided by earned premium.*
   - template: `SELECT incurred_losses / earned_premium AS ratio FROM cargo_experience WHERE line_of_business = :lob`;
   - properties used: `incurred_losses`, `earned_premium`, `line_of_business`.
4. **New entity through a second chunk.** Dual-ingest `doc_chunk_102`:
   - source `doc://underwriting/atlantic-freight-2026`;
   - text: *Underwriting update for Atlantic Freight: Marine cargo policy underwritten for transatlantic container shipments with piracy and war risk exclusions.*;
   - entity `ent_atlantic_freight`, named *Atlantic Freight*, type `MarineCargo`, confidence 0.97, with the marine-cargo `class_uri`;
   - pass `pg_store=pg_store` so both vector stores get it (goal 4 benchmarks both).
5. **Wait for both chunks.** Change `wait_until_searchable(expected_count=1)` to `expected_count=2`.

**Names must be exact**: `ent_atlantic_freight`, `MarineCargoPolicy` and `cargo_loss_ratio`. The evidence query, `metric_service.py` (`GOVERNED_METRIC_SQL`) and `assistant.py` (`LOB_BY_CLASS`) look them up by name.

In [ ]:
labkit.show('src/main.py', 'Ingesting ontology classes', before=1, after=62)

<details><summary>Hint 1: order matters</summary><code>ingest_governed_metric</code> <i>MATCH</i>es its target classes. If the class doesn't exist yet, the metric is created with no <code>COMPUTED_OVER</code> edge. Ingest the class before the metric.</details>
<details><summary>Hint 2: where the link comes from</summary>Look at the <code>extracted_entities</code> dict for Nexus Tower. Which key does the ingestion Cypher test before it writes <code>INSTANCE_OF</code>?</details>
<details><summary>Hint 3: the second chunk</summary>Copy the <code>doc_chunk_101</code> call and change the id, text, source and entity. Don't forget <code>pg_store=pg_store</code>.</details>

## Re-seed

Use the **seed-only** path. A full `python -m src.main` also runs the retrieval self-test, which keeps failing until goal 4 is fixed. Re-seeding is idempotent: everything is `MERGE` / upsert, and `s-demo` is rewritten with the same three turns.

In [ ]:
!{sys.executable} -m src.main --seed-only

In [ ]:
rows = {r['entity_id']: r for r in cypher(_ONTOLOGY_QUERY)}
expected = {'ent_nexus_tower': ('CommercialPropertyPolicy', ['solvency_ii_ratio']),
            'ent_atlantic_freight': ('MarineCargoPolicy', ['cargo_loss_ratio'])}
for ent, (cls, metrics) in expected.items():
    r = rows.get(ent, {})
    ok = r.get('class_name') == cls and r.get('applicable_metrics') == metrics
    print(f"{'OK  ' if ok else 'FAIL'} {ent:22} class={r.get('class_name')} metrics={r.get('applicable_metrics')}")

# The governed metrics now compute end-to-end over the PostgreSQL source of record
from src.pgvector_client import PgVectorStore
from src.metric_service import compute_governed_metric
pg = PgVectorStore()
print(compute_governed_metric(pg, 'solvency_ii_ratio', 'commercial_property'))   # 1.85
print(compute_governed_metric(pg, 'cargo_loss_ratio', 'marine_cargo'))           # 0.62
pg.close()

## Publish the evidence

In [ ]:
!{sys.executable} -m src.emit_evidence ontology
labkit.upload('ontology_result.json')
labkit.grade(3)

### Debrief

- **The ontology is the contract between unstructured text and governed numbers.** An entity without `INSTANCE_OF` is invisible to metric resolution, even though its text is perfectly searchable.
- **Governed metrics are stored as definitions** (template, properties, scope) in the graph. Execution uses only the allow-listed SQL in `metric_service.py`, with bound parameters and a read-only transaction. A new metric needs **both**: a graph definition and a vetted query.
- Run `!{sys.executable} -m src.emit_evidence preview` again: you should now see 2 entities, 3 classes, 2 metrics and 2 `INSTANCE_OF` links.